In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import numpy as np
import shapely

from gridr.core.grid import grid_commons
from gridr.core.grid import grid_rasterize
from gridr.core.grid import grid_mask

from notebook_utils import plot_convention_grid_mesh, mpl_plot_wrapper

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# Shared raster geometry used throughout the masking tutorial series.
shape, resolution = (6, 8), (1, 1)
origin_int, origin_half = (0., 0.), (0.5, 0.5)

# Grid-coordinate arrays for both conventions, used by plotting helpers.
cxx_int,  cyy_int  = grid_commons.grid_regular_coords_2d(shape, origin_int,  resolution, sparse=False)
cxx_half, cyy_half = grid_commons.grid_regular_coords_2d(shape, origin_half, resolution, sparse=False)

# Default value-to-style map for binary mask plots
# (used by notebooks 003+ which display GridR Validity rasters).
mask_value_color_alpha_map = (
    (grid_mask.Validity.VALID,   "orange", 0.2),
    (grid_mask.Validity.INVALID, "blue",   0.2),
)


# `build_mask` with all masking modes combined

This final notebook of the masking series combines everything covered so far: a valid geometry, one or more invalid geometries, and a low-resolution input raster mask. We then verify the result is exactly the binary `AND` of the per-mode outputs.

**What you'll learn**

- How the three masking modes combine inside `build_mask`
- The `AND` semantics: a pixel is valid only if it is valid in
  every mode that contributes to the call
- How to reproduce the combined mask manually from the per-mode masks

## Setting things up

We reuse the same inputs as the previous notebooks: the valid/invalid geometries of notebook 003 and the low-resolution input mask of notebook 004.

In [ ]:
rasterize_kwargs = {
    "alg": grid_rasterize.GridRasterizeAlg.RASTERIO_RASTERIZE,
    "kwargs_alg": {},
}

# Geometry inputs (notebook 003).
geometry_origin = (0.5, 0.5)
geometry_valid = shapely.geometry.Polygon([
    (3.5, 2.5),
    (6.5, 2.5),
    (6.5, 4.5),
    (3.5, 4.5),
])
geometry_invalid = [
    shapely.geometry.Polygon([
        (1.5, 1.5),
        (4.5, 1.5),
        (4.5, 2.5),
        (1.5, 2.5),
    ]),
    shapely.geometry.Polygon([
        (5.5, 3.5),
        (6.5, 3.5),
        (6.5, 4.5),
        (5.5, 4.5),
    ]),
]

# Input mask (notebook 004).
input_mask_lowres = np.full((3, 3), grid_mask.Validity.INVALID, dtype=np.uint8)
input_mask_lowres[0, 1] = grid_mask.Validity.VALID
input_mask_lowres[1, 1] = grid_mask.Validity.VALID
input_mask_lowres[1, 0] = grid_mask.Validity.VALID
input_mask_lowres[2, 1] = grid_mask.Validity.VALID
input_mask_lowres[2, 0] = grid_mask.Validity.VALID
input_mask_lowres_resolution = (4, 4)

## Combined `build_mask` call

In [ ]:
raster = grid_mask.build_mask(
    shape                    = shape,
    resolution               = (1, 1),
    out                      = None,
    geometry_origin          = geometry_origin,
    geometry_pair            = [geometry_valid, geometry_invalid],
    mask_in                  = input_mask_lowres,
    mask_in_target_win       = None,
    mask_in_resolution       = input_mask_lowres_resolution,
    oversampling_dtype       = np.float64,
    mask_in_binary_threshold = 0.99,
    rasterize_kwargs         = rasterize_kwargs,
)

In [ ]:
plot_convention_grid_mesh(
    shape, resolution, origin_int, cxx_int, cyy_int,
    geometry=None, geometry_origin=None,
    mask=raster, win=None,
    value_color_alpha_map=mask_value_color_alpha_map,
    prefix="build_mask_all_modes",
    title="Build mask with all masking modes",
)

Only three pixels remain valid: those that survive **all three masking modes simultaneously**. The combination rule is the same one we saw between valid and invalid geometries -- a binary `AND` across modes.

## Verifying the AND combination

For full transparency, we recompute the per-mode masks separately and verify that their binary `AND` equals the combined `build_mask` output.

### Mask from geometries only

In [ ]:
raster_geom = grid_mask.build_mask(
    shape            = shape,
    resolution       = (1, 1),
    out              = None,
    geometry_origin  = geometry_origin,
    geometry_pair    = [geometry_valid, geometry_invalid],
    rasterize_kwargs = rasterize_kwargs,
)
print(raster_geom)

### Mask from input raster only

In [ ]:
raster_mask_in = grid_mask.build_mask(
    shape                    = shape,
    resolution               = (1, 1),
    out                      = None,
    mask_in                  = input_mask_lowres,
    mask_in_target_win       = None,
    mask_in_resolution       = input_mask_lowres_resolution,
    oversampling_dtype       = np.float64,
    mask_in_binary_threshold = 0.99,
)
print(raster_mask_in)

### Binary AND merge

In [ ]:
print(raster_geom & raster_mask_in)

This is exactly the combined `build_mask` output above, confirming the documented merge semantics.